# Parameter Sensitivity Analysis

Loads `experiments/results.csv` and tests whether observed differences across
parameter levels are statistically significant.

**Sweeps**
- **A** — generations (5, 10, 20, 50, 100), `population_size=200`
- **B** — population size (50, 100, 200, 500, 1000), `generations=10`
- **C** — fitness weight presets (optional)

**Statistical tests**
- `best_fitness` (continuous): Kruskal-Wallis H; post-hoc Dunn with Holm correction
- `implies_ideal` (binary, 1 = repair is equivalent or stronger than an ideal):
  chi-square; post-hoc Fisher's exact with Bonferroni correction


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import scikit_posthocs as sp
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')
sns.set_palette('tab10')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.4f}'.format)


In [ ]:
results_path = Path('../experiments/results.csv')
if not results_path.exists():
    raise FileNotFoundError(
        f'Results not found: {results_path}  --  run: python scripts/run_experiments.py'
    )

df = pd.read_csv(results_path)
# Coerce level_value to numeric where possible (sweeps A/B have integer levels)
df['level_value'] = pd.to_numeric(df['level_value'], errors='coerce').fillna(df['level_value'])

print(f'Loaded {len(df)} rows')
print(df.groupby(['sweep', 'spec'])['seed'].count().rename('n_runs').to_string())
df.head()


In [ ]:
def analyse_sweep(df, sweep, xlabel, sort_numeric=True):
    sdf = df[df['sweep'] == sweep].copy()
    if sdf.empty:
        print(f'No data for sweep {sweep}')
        return

    # Determine display order for levels
    lv = sdf.drop_duplicates('level_name')[['level_name', 'level_value']].copy()
    if sort_numeric:
        lv['_n'] = pd.to_numeric(lv['level_value'], errors='coerce')
        lv = lv.sort_values('_n')
    order = lv['level_name'].tolist()

    for spec in sorted(sdf['spec'].unique()):
        spec_df = sdf[sdf['spec'] == spec].copy()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Sweep {sweep} - {spec}  ({xlabel})', fontsize=13)

        # Panel 1: best_fitness distribution
        valid = spec_df[spec_df['best_fitness'].notna()]
        if not valid.empty:
            sns.boxplot(data=valid, x='level_name', y='best_fitness', order=order, ax=axes[0])
        axes[0].set(title='Best fitness score', xlabel=xlabel, ylabel='Fitness')
        axes[0].tick_params(axis='x', rotation=45)

        # Panel 2: implies_ideal rate
        rates = spec_df.groupby('level_name')['implies_ideal'].mean().reindex(order).fillna(0)
        axes[1].bar(range(len(order)), rates.values, color='steelblue')
        axes[1].set_xticks(range(len(order)))
        axes[1].set_xticklabels(order, rotation=45)
        axes[1].set(title='Rate: repair implies ideal', xlabel=xlabel, ylabel='Rate', ylim=(0, 1.05))

        # Panel 3: found_repair rate
        found = spec_df.groupby('level_name')['found_repair'].mean().reindex(order).fillna(0)
        axes[2].bar(range(len(order)), found.values, color='darkorange')
        axes[2].set_xticks(range(len(order)))
        axes[2].set_xticklabels(order, rotation=45)
        axes[2].set(title='Rate: any repair found', xlabel=xlabel, ylabel='Rate', ylim=(0, 1.05))

        plt.tight_layout()
        plt.show()

        print()
        print(f'=== Sweep {sweep} / {spec} ===')

        # --- Kruskal-Wallis on best_fitness ---
        fit_groups = [
            spec_df[spec_df['level_name'] == ln]['best_fitness'].dropna().values
            for ln in order
        ]
        fit_groups = [g for g in fit_groups if len(g) > 0]
        if len(fit_groups) >= 2:
            H, p_kw = stats.kruskal(*fit_groups)
            n = int(spec_df['best_fitness'].notna().sum())
            k = len(fit_groups)
            eps2 = (H - k + 1) / (n - k) if n > k else float('nan')
            sig = ' ***' if p_kw < 0.001 else ' **' if p_kw < 0.01 else ' *' if p_kw < 0.05 else ' ns'
            print(f'  Kruskal-Wallis (best_fitness):  H={H:.3f}  p={p_kw:.4f}{sig}  eps2={eps2:.3f}')
            if p_kw < 0.05:
                valid_df = spec_df[spec_df['best_fitness'].notna()]
                dunn = sp.posthoc_dunn(
                    valid_df, val_col='best_fitness',
                    group_col='level_name', p_adjust='holm'
                )
                present = [l for l in order if l in dunn.index]
                print('  Post-hoc Dunn (Holm correction):')
                print(dunn.reindex(index=present, columns=present).round(4).to_string())

        # --- Chi-square on implies_ideal ---
        ct = pd.crosstab(spec_df['level_name'], spec_df['implies_ideal'])
        ct = ct.reindex(order).fillna(0).astype(int)
        if 0 in ct.columns and 1 in ct.columns and len(ct) >= 2:
            chi2_stat, p_chi, dof, _ = stats.chi2_contingency(ct)
            sig = ' ***' if p_chi < 0.001 else ' **' if p_chi < 0.01 else ' *' if p_chi < 0.05 else ' ns'
            print(f'  Chi-square (implies_ideal):     chi2={chi2_stat:.3f}  p={p_chi:.4f}{sig}  dof={dof}')
            if p_chi < 0.05:
                pairs = []
                for i, a in enumerate(order):
                    for b in order[i + 1:]:
                        a1 = int(ct.loc[a, 1]) if a in ct.index else 0
                        a0 = int(ct.loc[a, 0]) if a in ct.index else 0
                        b1 = int(ct.loc[b, 1]) if b in ct.index else 0
                        b0 = int(ct.loc[b, 0]) if b in ct.index else 0
                        _, pv = stats.fisher_exact([[a1, a0], [b1, b0]])
                        pairs.append((a, b, pv))
                n_pairs = len(pairs)
                for a, b, pv in sorted(pairs, key=lambda x: x[2]):
                    padj = min(pv * n_pairs, 1.0)
                    mark = ' *' if padj < 0.05 else ''
                    print(f'    {a} vs {b}:  p_raw={pv:.4f}  p_adj={padj:.4f}{mark}')
        print()


## Sweep A — Generations

Levels: 5, 10, 20, 50, 100.  `population_size=200`, default fitness weights.


In [ ]:
analyse_sweep(df, 'A', 'Generations', sort_numeric=True)


## Sweep B — Population Size

Levels: 50, 100, 200, 500, 1000.  `generations=10`, default fitness weights.


In [ ]:
analyse_sweep(df, 'B', 'Population Size', sort_numeric=True)


## Sweep C — Fitness Weight Presets

Optional sweep.  Levels: `default`, `syntactic-heavy`, `semantic-heavy`,
`status-only`, `no-halstead`.  Run with `python scripts/run_experiments.py --sweeps C`.


In [ ]:
if 'C' in df['sweep'].values:
    analyse_sweep(df, 'C', 'Weight Preset', sort_numeric=False)
else:
    print('Sweep C not in results.csv.')
    print('Run: python scripts/run_experiments.py --sweeps C')


## Cross-Sweep Summary


In [ ]:
summary = (
    df.groupby(['sweep', 'level_name', 'spec'])
    .agg(
        n_runs=('seed', 'count'),
        found_repair_rate=('found_repair', 'mean'),
        implies_ideal_rate=('implies_ideal', 'mean'),
        median_fitness=('best_fitness', 'median'),
        mean_wall_s=('wall_time_s', 'mean'),
    )
    .round(3)
)
summary
